# Plot the results of synthetic slip inversion based on ANY model of $\mu$ using the synthetic displacement based on ANY model of $\mu$ from a prescribed, ground-truth slip distribution on the fault

* **ANY** means the $\mu$ structure can be either homogeneous or heterogeneous.

* Once we know which elastic property the ground displacement is sensitive to, we may proceed with the inversion.

* The most basic thing to check is if you can recover the ground truth slip and fit the observation under a heterogeneous model of $\mu$. In theory, this should not be different from the homogeneous case, to make sure the code is running properly, you need to pass this test.

* The next level is how different the inferred slip distributions are under homogeneous and heterogeneous cases. The used ground displacement can be from either model, but maybe from heterogeneous $\mu$ structure should be more emphasized.


In [ ]:
# Load libraries
import numpy as np
import pygmt
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import rc
from io import StringIO
import utils_plot as utp

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

In [ ]:
# work dir
datadir = "/home/staff/chao/SSEinv/Nicoya/"

# Define the inversion results path
resultpath = "rst_coseis/"

# coseismic data file
obs_disp_name = "data/Protti_etal_2014_tableS1.csv" 

# the processed data has the unit of m/yr that was converted from mm/yr
df = pd.read_csv(datadir + obs_disp_name, sep=",", skiprows=1, \
                 names=['site', 'lon', 'lat', 'elv', 'ux', 'uy', 'uz', \
                        'ux_std', 'uy_std', 'uz_std', 'date0', 'date1', 'duration'])

In [ ]:
# Decide the weights of the horizontal, vertical components
# f_h, f_v = 1, 1/2
f_h, f_v = 1, 1
# Print the weights of the data
print( "Data weight horizontal / vertical: %.2f / %.2f" %(f_h, f_v) )
# Take the inverse for saving the name of the weights
w_h, w_v = int(1/f_h), int(1/f_v)

# Define the reference point (center of the projection)
lon0, lat0 = -84, 7     # from Christos's email

In [ ]:
# meshnameCK = "nicoyaCK3"   # fault zone extended to the whole subduction zone
# meshnameCK = "nicoyaCK4"   # same as CK2, but connecting the trench now

# meshnameCK = "nicoyaCKden_sm"   # based on nicoyaCK3 or 4, but denser mesh size, and smaller fault zone
meshnameCK = "nicoyaCKden_all"   # based on nicoyaCK3 or 4, but denser mesh size, and all subduction interface

def mu_expression(m):
    mu = 20*(2. + np.tanh(m))
    return mu

# background shear modulus
mu_b = 0   # 40 GPa
mu_background = mu_expression(mu_b)

# shear modulus for the lower (subducting) plate
mu_l = 0.9730 # ~55 GPa
mu_lower = mu_expression(mu_l)

# shear modulus for the upper (overriding) plate
mu_u = -0.9730  # ~25 GPa
# mu_u = mu_b
mu_upper = mu_expression(mu_u)

# string for the homogeneous solution
homo_str = f"_mul{round(mu_expression(mu_b))}u{round(mu_expression(mu_b))}"
# string for the contrast between slab and wedge
sw_str  = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}"
# string for the 3d model, value multiplied by 4, relative a reference
# contrast_factor = 4.0  # amplification factor 
contrast_factor = 1.0  # amplification factor 
het3d_str = f"_DeShon3D_ref_{round(contrast_factor)}"

rho_s = 2e7   # allows variations of slip of the order of ~4.5 km, close to the maximum resolution

# L-curve, CK mesh, inversion in HOM model
# gammas_CK_hom = [2e1, 4e1, 8e1, 1e2, 2e2, 4e2, 6e2, 8e2, 2e3, 1e4]
gammas_CK_hom = [2e1, 4e1, 8e1, 1e2, 2e2, 4e2, 5e2, 6e2, 7e2, 8e2, 9e2, 2e3, 1e4]   # extra added
outFileName = f"Lcurvecoseis_rs{rho_s:.0e}_{meshnameCK}_{homo_str}.txt"
misfitsCK_hom = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis'])

# L-curve, CK mesh, inversion in SW model
# gammas_CK_sw = [2e1, 4e1, 8e1, 1e2, 2e2, 4e2, 6e2, 8e2, 2e3, 1e4]
gammas_CK_sw = [2e1, 4e1, 8e1, 1e2, 2e2, 4e2, 5e2, 6e2, 7e2, 8e2, 9e2, 2e3, 1e4]   # extra added
outFileName = f"Lcurvecoseis_rs{rho_s:.0e}_{meshnameCK}_{sw_str}.txt"
misfitsCK_sw = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis'])

# L-curve, CK mesh, inversion in original 3D model
# gammas_CK_3d = [2e1, 4e1, 8e1, 1e2, 2e2, 4e2, 6e2, 8e2, 2e3, 1e4]
gammas_CK_3d = [2e1, 4e1, 8e1, 1e2, 2e2, 4e2, 5e2, 6e2, 7e2, 8e2, 9e2, 2e3, 1e4]   # extra added
outFileName = f"Lcurvecoseis_rs{rho_s:.0e}_{meshnameCK}_{het3d_str}.txt"
misfitsCK_3d = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis'])


In [ ]:
print(gammas_CK_hom)
print(gammas_CK_hom[1:])


In [ ]:
gamma_CK_hom_prefer = 6e2
idxCK_hom = gammas_CK_hom.index(gamma_CK_hom_prefer)
print(idxCK_hom, gammas_CK_hom[idxCK_hom])

gamma_CK_sw_prefer = 6e2
idxCK_sw = gammas_CK_sw.index(gamma_CK_hom_prefer)
print(idxCK_sw, gammas_CK_sw[idxCK_sw])

gamma_CK_3d_prefer = 6e2
idxCK_3d = gammas_CK_3d.index(gamma_CK_3d_prefer)
print(idxCK_3d, gammas_CK_3d[idxCK_3d])

In [ ]:
# Plot L-curve
fig, axes = plt.subplots(1, 1, figsize=(4,4), dpi=300)

ax = axes
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=8)
ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=8)
ax.set_title("Coseismic slip inversion", fontsize=9)
ax.tick_params(labelsize=8)

color_L = ['silver', 'firebrick', 'white']

ax.plot(misfitsCK_hom['m_mis'][1:], misfitsCK_hom['d_mis'][1:], 
        'o-', color='k', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"HOM")
# idxCK_hom = 6
ax.plot(misfitsCK_hom.iloc[idxCK_hom]['m_mis'], misfitsCK_hom.iloc[idxCK_hom]['d_mis'], 
        'o', color='k', markerfacecolor='k', markersize=5, 
        label=r"HOM; preferred $\gamma$={:.1e}".format(gammas_CK_hom[idxCK_hom]))

ax.plot(misfitsCK_sw['m_mis'][1:], misfitsCK_sw['d_mis'][1:], 
        's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"SW")
# idxCK_sw = 6
ax.plot(misfitsCK_sw.iloc[idxCK_sw]['m_mis'], misfitsCK_sw.iloc[idxCK_sw]['d_mis'], 
        's', color='red', markerfacecolor='red', markersize=5, 
        label=r"SW; preferred $\gamma$={:.1e}".format(gammas_CK_sw[idxCK_sw]))

ax.plot(misfitsCK_3d['m_mis'][1:], misfitsCK_3d['d_mis'][1:], 
        'D-', color='blue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"3D")
# idxCK_3d = 6
ax.plot(misfitsCK_3d.iloc[idxCK_3d]['m_mis'], misfitsCK_3d.iloc[idxCK_3d]['d_mis'], 
        'D', color='blue', markerfacecolor='blue', markersize=5, 
        label=r"3D; preferred $\gamma$={:.1e}".format(gammas_CK_3d[idxCK_3d]))


# for i, gamma in enumerate(gammas_s):
#     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)
# ax.set_ylim(-5, 80)
# ax.set_xlim(0, 0.1)


figpath = datadir + '/figures/coseis/'
output_file = figpath + f'Lcurvecoseis_{meshnameCK}.pdf'
# plt.savefig(output_file, dpi=300, bbox_inches='tight')

In [ ]:
meshnameCK = 'nicoyaCKden_une_all'

homo_str = f"_mul{round(mu_expression(mu_b))}u{round(mu_expression(mu_b))}"
sw_str  = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}"
het3d_str_ori = f"_DeShon3D_ref_{round(1.0)}_hull"
contrast_factor = 2.5  # adopted since 03/05/2026
het3d_str     = f"_DeShon3D_ref1D_{round(contrast_factor)}_hull"

CG_mu_deg = 2   # 1 for hom or SW, 2 for 3D
if CG_mu_deg == 2:
    het3d_str_ori = het3d_str_ori + f"_CG{CG_mu_deg}"
    het3d_str     = het3d_str     + f"_CG{CG_mu_deg}"

def load_lcurve(rho_s, meshname, mu_str):
    rspath = datadir + resultpath
    fname = f"Lcurvecoseis_rs{rho_s:.0e}_{meshname}_{mu_str}.txt"
    df = pd.read_csv(rspath + fname, sep=r'\s+', names=['d_mis', 'm_mis', 'gamma', 'rho_s'])
    return df.drop_duplicates(subset='gamma').sort_values('gamma').reset_index(drop=True)

def _idx_for_gamma_co(df, gamma_pref):
    return int((df['gamma'] - gamma_pref).abs().idxmin())

rho_s = 2e7

misfitsCK_hom = load_lcurve(rho_s, meshnameCK, homo_str)
misfitsCK_sw = load_lcurve(rho_s, meshnameCK, sw_str)
misfitsCK_3d_ori = load_lcurve(rho_s, meshnameCK, het3d_str_ori)
misfitsCK_3d = load_lcurve(rho_s, meshnameCK, het3d_str)

gamma_CK_hom_prefer    = 2.5e2
gamma_CK_3d_ori_prefer = 2.5e2
gamma_CK_sw_prefer     = 3e2
gamma_CK_3d_prefer     = 3e2

idxCK_hom_co    = _idx_for_gamma_co(misfitsCK_hom,    gamma_CK_hom_prefer)
idxCK_sw_co     = _idx_for_gamma_co(misfitsCK_sw,     gamma_CK_sw_prefer)
idxCK_3d_ori_co = _idx_for_gamma_co(misfitsCK_3d_ori, gamma_CK_3d_ori_prefer)
idxCK_3d_co     = _idx_for_gamma_co(misfitsCK_3d,     gamma_CK_3d_prefer)

# Plot L-curve
fig, axes = plt.subplots(1, 1, figsize=(4,4), dpi=300)

ax = axes
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=8)
ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=8)
ax.set_title("Real coseismic slip inversion", fontsize=9)
ax.tick_params(labelsize=8)

color_L = ['silver', 'firebrick', 'white']

i_start, i_end = 0, -5
df_plot = misfitsCK_hom.iloc[i_start:i_end]
ax.plot(df_plot['m_mis'], df_plot['d_mis'],
        'o-', color='k', markerfacecolor=color_L[2],
        linewidth=1.0, markersize=4, label=r"H")
ax.plot(misfitsCK_hom.iloc[idxCK_hom_co]['m_mis'], misfitsCK_hom.iloc[idxCK_hom_co]['d_mis'], 
        'o', color='k', markerfacecolor='k', markersize=5, 
        label=r"H preferred $\gamma$={:.1e}".format(misfitsCK_hom['gamma'].iloc[idxCK_hom_co]))

for _, row in df_plot.iterrows():
    ax.text(row['m_mis']+0.25, row['d_mis'], f"{row['gamma']:.0f}", fontsize=6, color='k')

df_plot = misfitsCK_sw.iloc[i_start:i_end]
ax.plot(df_plot['m_mis'], df_plot['d_mis'],
        's-', color='red', markerfacecolor=color_L[2], 
        linewidth=1.0, markersize=4, label=r"S")
ax.plot(misfitsCK_sw.iloc[idxCK_sw_co]['m_mis'], misfitsCK_sw.iloc[idxCK_sw_co]['d_mis'], 
        's', color='red', markerfacecolor='red', markersize=5, 
        label=r"S preferred $\gamma$={:.1e}".format(misfitsCK_sw['gamma'].iloc[idxCK_sw_co]))

df_plot = misfitsCK_3d.iloc[i_start:i_end]
ax.plot(df_plot['m_mis'], df_plot['d_mis'],
        'D-', color='dodgerblue', markerfacecolor=color_L[2], 
        linewidth=1.0, markersize=4, label=r"3D")
ax.plot(misfitsCK_3d.iloc[idxCK_3d_co]['m_mis'], misfitsCK_3d.iloc[idxCK_3d_co]['d_mis'], 
        'D', color='dodgerblue', markerfacecolor='dodgerblue', markersize=5, 
        label=r"3D preferred $\gamma$={:.1e}".format(misfitsCK_3d['gamma'].iloc[idxCK_3d_co]))

df_plot = misfitsCK_3d_ori.iloc[i_start:i_end]
ax.plot(df_plot['m_mis'], df_plot['d_mis'],
        'D-', color='darkblue', markerfacecolor=color_L[2], 
        linewidth=1.0, markersize=4, label=r"Orig. 3D")
ax.plot(misfitsCK_3d_ori.iloc[idxCK_3d_ori_co]['m_mis'], misfitsCK_3d_ori.iloc[idxCK_3d_ori_co]['d_mis'], 
        'D', color='darkblue', markerfacecolor='darkblue', markersize=5, 
        label=r"Orig. 3D preferred $\gamma$={:.1e}".format(misfitsCK_3d_ori['gamma'].iloc[idxCK_3d_ori_co]))


ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=8)
# ax.set_ylim(0, 0.6)
# ax.set_xlim(0, 0.1)


In [ ]:
meshnameCK = 'nicoyaCKden_une_all'
rho_s = 2e7

# mu model strings (homo_str, sw_str inherited from earlier cells)
het3d_str     = "_DeShon3D_ref1D_2_hull_CG2"   # contrast_factor=2.5, ref1D, hull
het3d_ori_str = "_DeShon3D_ref_1_hull_CG2"     # contrast_factor=1.0, ref global mean, hull

def load_lcurve(fname):
    df = pd.read_csv(fname, sep=r'\s+', names=['d_mis', 'm_mis', 'gamma', 'rho_s'])
    return df.sort_values('gamma').reset_index(drop=True)

rspath = datadir + resultpath
misfits_hom    = load_lcurve(rspath + f"Lcurvecoseis_rs{rho_s:.0e}_{meshnameCK}_{homo_str}.txt")
misfits_sw     = load_lcurve(rspath + f"Lcurvecoseis_rs{rho_s:.0e}_{meshnameCK}_{sw_str}.txt")
misfits_3d     = load_lcurve(rspath + f"Lcurvecoseis_rs{rho_s:.0e}_{meshnameCK}_{het3d_str}.txt")
misfits_3d_ori = load_lcurve(rspath + f"Lcurvecoseis_rs{rho_s:.0e}_{meshnameCK}_{het3d_ori_str}.txt")

print("HOM gammas:", misfits_hom['gamma'].tolist())
print("SW  gammas:", misfits_sw['gamma'].tolist())
print("3D  gammas:", misfits_3d['gamma'].tolist())
print("Ori gammas:", misfits_3d_ori['gamma'].tolist())

# Preferred gammas (adjust after inspecting the L-curves)
gamma_hom_prefer    = 2.5e2
gamma_3d_ori_prefer = 2.5e2
gamma_sw_prefer     = 3e2
gamma_3d_prefer     = 3e2
    
curves = [
    (misfits_hom,    "HOM",       'k',      'o', gamma_hom_prefer),
    (misfits_sw,     "Het K2L",   'red',    's', gamma_sw_prefer),
    (misfits_3d,     "3D Het",    'blue',   'D', gamma_3d_prefer),
    (misfits_3d_ori, "Orig. 3D",  'purple', '^', gamma_3d_ori_prefer),
]

fig, ax = plt.subplots(1, 1, figsize=(5, 5), dpi=300)
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=8)
ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=8)
ax.set_title(f"Coseismic L-curve — {meshnameCK}\n$\\rho_s$={rho_s:.0e}", fontsize=9)
ax.tick_params(labelsize=8)

for df, label, color, marker, gamma_pref in curves:
    ax.plot(df['m_mis'], df['d_mis'], f'{marker}-', color=color,
            markerfacecolor='white', linewidth=1.0, markersize=4, label=label)
    pref = df[df['gamma'] == gamma_pref]
    if len(pref):
        ax.plot(pref.iloc[0]['m_mis'], pref.iloc[0]['d_mis'], marker,
                color=color, markerfacecolor=color, markersize=4,
                label=f"{label}; preferred $\gamma$={gamma_pref:.1e}")

# Annotate γ values only on the HOM curve to keep the figure uncluttered —
# all four curves share essentially the same γ grid, so labelling HOM is enough.
# Pixel-space offset (`textcoords='offset points'`) keeps the label a constant
# small distance NE of each marker regardless of axis scale or zoom.
for _, row in misfits_hom.iterrows():
    ax.annotate(f"{row['gamma']:.0f}",
                xy=(row['m_mis'], row['d_mis']),
                xytext=(2, 2), textcoords='offset points',
                fontsize=6, color='k', ha='left', va='bottom')

ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6)
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2)
ax.set_box_aspect(1)
ax.legend(fontsize=7)

figpath = datadir + '/figures/coseis/'
output_file = figpath + f'Lcurvecoseis_rs{rho_s:.0e}_{meshnameCK}_all.pdf'
# plt.savefig(output_file, dpi=300, bbox_inches='tight')


In [ ]:
rho_s1 = 2.5e8   # allows variations of slip of the order of ~15 km
outFileName = f"Lcurvecoseis_rs{rho_s1:.1e}_{meshnameCK}_{homo_str}.txt"
misfitsCK_hom1 = load_lcurve(rho_s1, meshnameCK, homo_str)

rho_s2 = 1e9   # allows variations of slip of the order of ~30 km
outFileName = f"Lcurvecoseis_rs{rho_s2:.1e}_{meshnameCK}_{homo_str}.txt"
misfitsCK_hom2 = load_lcurve(rho_s2, meshnameCK, homo_str)

# L-curve, CK mesh, inversion in SW model
gammas_CK_sw1 = [8e1, 2e2, 4e2, 6e2, 8e2, 2e3]   # extra added
outFileName = f"Lcurvecoseis_rs{rho_s1:.1e}_{meshnameCK}_{sw_str}.txt"
misfitsCK_sw1 = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis', 'gamma', 'rho_s']) 

het3d_str = f"_DeShon3D_ref1D_{round(1.0)}_hull"   # ref 1D value at each depth layer
CG_mu_deg = 2   # 1 for hom or SW, 2 for 3D
if CG_mu_deg == 2:
    het3d_str = het3d_str + f"_CG{CG_mu_deg}"

# L-curve, CK mesh, inversion in original 3D model
gammas_CK_3d1 = [8e1, 2e2, 4e2, 6e2, 8e2, 2e3]   # extra added
outFileName = f"Lcurvecoseis_rs{rho_s1:.1e}_{meshnameCK}_{het3d_str}.txt"
misfitsCK_3d1 = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis', 'gamma', 'rho_s']) 

# Plot L-curve
fig, axes = plt.subplots(1, 1, figsize=(4,4), dpi=300)

ax = axes
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=8)
ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=8)
ax.set_title("Coseismic slip inversion", fontsize=9)
ax.tick_params(labelsize=8)

color_L = ['silver', 'firebrick', 'white']

i_start, i_end = 0, None
# ax.plot(misfitsCK_hom1['m_mis'][i_start:i_end], misfitsCK_hom1['d_mis'][i_start:i_end], 
#         'o-', color='b', markerfacecolor=color_L[2],
#         linewidth=1.0, markersize=4, label=r"HOM; $\rho$={:.1e}".format(rho_s1))

ax.plot(misfitsCK_hom1['m_mis'][i_start:i_end], misfitsCK_hom1['d_mis'][i_start:i_end], 
        'o-', color='k', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, 
        label=r"HOM; $\rho$={:.1e}".format(rho_s1))
for i, gamma in enumerate(gammas_CK_hom1[i_start:i_end]):
    ax.text(misfitsCK_hom1['m_mis'][i_start:i_end].iloc[i]+0.25, 
            misfitsCK_hom1['d_mis'][i_start:i_end].iloc[i], f"{gamma:.0f}", fontsize=6, color='k')

ax.plot(misfitsCK_sw1['m_mis'][i_start:i_end], misfitsCK_sw1['d_mis'][i_start:i_end], 
        's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, 
        label=r"SW; $\rho$={:.1e}".format(rho_s1))

ax.plot(misfitsCK_3d1['m_mis'][i_start:i_end], misfitsCK_3d1['d_mis'][i_start:i_end], 
        'D-', color='blue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, 
        label=r"Orig. 3D; $\rho$={:.1e}".format(rho_s1))

# i_start, i_end = 1, None
# ax.plot(misfitsCK_hom['m_mis'][i_start:i_end], misfitsCK_hom['d_mis'][i_start:i_end], 
#         'o--', color='k', markerfacecolor=color_L[2],
#         linewidth=1.0, markersize=4, label=r"HOM; $\rho$={:.1e}".format(rho_s))
# # r"HOM; preferred $\gamma$={:.1e}".format(gammas_CK_hom[idxCK_hom])
# for i, gamma in enumerate(gammas_CK_hom[i_start:i_end]):
#     ax.text(misfitsCK_hom['m_mis'][i_start:i_end].iloc[i]+0.25, 
#             misfitsCK_hom['d_mis'][i_start:i_end].iloc[i], f"{gamma:.0f}", fontsize=6, color='k')
# ax.text
# ax.plot(misfitsCK_swsw['m_mis'], misfitsCK_swsw['d_mis'], 's-', color='red', markerfacecolor=color_L[2],
#         linewidth=1.0, markersize=4, label=r"SW; SW")
# ax.plot(misfitsCK_swhom['m_mis'], misfitsCK_swhom['d_mis'], 's--', color='red', markerfacecolor=color_L[0],
#         linewidth=1.0, markersize=4, label=r"SW; HOM;")
# ax.plot(misfitsCK_3d3d['m_mis'], misfitsCK_3d3d['d_mis'], 'D-', color='dodgerblue', markerfacecolor=color_L[2],
#         linewidth=1.0, markersize=4, label=r"3D; 3D")
# ax.plot(misfitsCK_3dhom['m_mis'], misfitsCK_3dhom['d_mis'], 'D--', color='dodgerblue', markerfacecolor=color_L[0],
#         linewidth=1.0, markersize=4, label=r"3D; HOM;")

ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=8)
# ax.set_ylim(0, 0.6)
ax.set_xlim(0, 175)


# Load the L-curve with $\rho$ fixed 

In [ ]:
# meshnameCK = "nicoyaCK"   # local interface model from C. Kyriakopoulos_etal2015JGRSE
# meshnameCK = "nicoyaCK2"   # same as above but 5-km mesh size on fault
meshnameCK = "nicoyaCK3"   # fault zone extended to the whole subduction zone
# meshnameCK = "nicoyaCK4"   # same as CK2, but connecting the trench now
print(meshnameCK)

# meshname = "nicoya"
# meshname = "nicoya2"   # This has a smaller fault interface
# meshname = "nicoya3"   # the same as above but 5-km mesh size on fault
meshname = "nicoya4"   # extended fault area
print(meshname)

# if meshnameCK == "nicoyaCK3":   # fault zone extended to the whole subduction zone

rho_s = 2e7   # allows variations of slip of the order of ~4.5 km, close to the maximum resolution
gammas_s = [2.5e1, 5e1, 7.5e1, 1e2, 2.5e2, 5e2, 1e3, 5e3, 1e4]

outFileName = f"Lcurvecoseisrs{rho_s:.0e}{meshnameCK}.txt"
misfitsCK = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis'])
gammas_s_used = 1e2
idxCK = gammas_s.index(gammas_s_used)
print(idxCK)
print(gammas_s[idxCK])

outFileName = f"Lcurvecoseisrs{rho_s:.0e}{meshname}.txt"
misfits = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis'])
gammas_s_used = 1e2  
idx = gammas_s.index(gammas_s_used)
print(gammas_s[idx])

### L-curve criterion ###
color_L = ['silver', 'firebrick']

# Plot L-curve
fig, axes = plt.subplots(1,1,figsize=(4,4), dpi=300)

# ax = axes[0]
ax = axes
ax.set_xlabel(r"$||\mathbf{Gm} - \mathbf{d}||$", fontsize=8)
ax.set_ylabel(r"$||\mathbf{Lm}||$", fontsize=8)
ax.set_title("Coseismic slip inversion", fontsize=9)
ax.tick_params(labelsize=8)

ax.plot(misfitsCK['d_mis'], misfitsCK['m_mis'], 'o-', color='blue', markerfacecolor=color_L[0],
        linewidth=1.0, markersize=4, label=r"CK; $\rho$ = {:.1e}".format(rho_s))
ax.plot(misfitsCK.iloc[idxCK]['d_mis'], misfitsCK.iloc[idxCK]['m_mis'], 's', color='blue', markerfacecolor='blue',
        markersize=5, label=r"CK; preferred $\gamma$ = {:.1e}".format(gammas_s[idxCK]))
ax.plot(misfits['d_mis'], misfits['m_mis'], 'o-', color='red', markerfacecolor=color_L[0],
        linewidth=1.0, markersize=4, label=r"Slab2; $\rho$ = {:.1e}".format(rho_s))
ax.plot(misfits.iloc[idx]['d_mis'], misfits.iloc[idx]['m_mis'], 's', color='red', markerfacecolor='red',
        markersize=5, label=r"Slab2; preferred $\gamma$ = {:.1e}".format(gammas_s[idx]))
# for i, gamma in enumerate(gammas_s):
#     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=8)